# Elden Ring Build Optimizer

This project implements an **AI agent** that helps players construct optimized character builds in Elden Ring (including the Shadow of the Erdtree DLC).

Rather than relying on a language model's training knowledge, the agent queries a structured dataset scraped from the Elden Ring wiki — covering weapons, upgrade paths, Ashes of War, talismans, armor, and spells — to return **data-backed, verifiable build recommendations**.

The agent uses the Anthropic API's tool use feature to decide which files to query, what filters to apply, and how to reason across multiple data sources to satisfy a user's build goal.

**Dataset:** [Ultimate Elden Ring with Shadow of the Erdtree DLC](https://www.kaggle.com/datasets/pedroaltobelli/ultimate-elden-ring-with-shadow-of-the-erdtree-dlc) by Pedro Altobelli (CC0 Public Domain)

**Stack:** Python, Pandas, Anthropic SDK

In [6]:
# Imports
import pandas as pd
import json
# import ast
import os


In [7]:
# Force kagglehub to use your Z: drive as cache
os.environ["KAGGLEHUB_CACHE"] = "Z:/kagglehub"

In [ ]:
# Grab the dataset from Kaggle using the kagglehub library
import kagglehub

path = kagglehub.dataset_download("pedroaltobelli/ultimate-elden-ring-with-shadow-of-the-erdtree-dlc")
BASE = path + '/eldenringScrap/'

print("Path to dataset files:", BASE)

100%|██████████| 1.88M/1.88M [00:00<00:00, 16.5MB/s]

Extracting files...
Path to dataset files:

 Z:/kagglehub\datasets\pedroaltobelli\ultimate-elden-ring-with-shadow-of-the-erdtree-dlc\versions\1/eldenringScrap/


## Section 1: Exploratory Data Analysis

Before defining any agent tools, we need to understand the structure of each CSV file.
This includes column names, data types, sample values, and any cleaning needed. 
This informs exactly what filters and queries our tools will support.

In [9]:
# Load the other core files for the build optimizer
# BASE = 'Z:/kagglehub/datasets/pedroaltobelli/ultimate-elden-ring-with-shadow-of-the-erdtree-dlc/versions/1/eldenringScrap/'

weapons = pd.read_csv(BASE + 'weapons.csv')
print(weapons.shape)
print(weapons.columns.tolist())
weapons.head()

(402, 13)
['id', 'weapon_id', 'name', 'image', 'weight', 'description', 'dlc', 'requirements', 'damage type', 'category', 'passive effect', 'skill', 'FP cost']


,id,weapon_id,name,image,weight,description,dlc,requirements,damage type,category,passive effect,skill,FP cost
0,0,1,Dueling Shield,http://eldenring.wiki.fextralife.com/file/Elde...,9.0,"A ""thrusting shield,"" or combined weapon and s...",1,"{'Str': 15, 'Dex': 14}",Standard/Pierce,Thrusting Shields,No passive effects,Shield Strike,0
1,1,2,Carian Thrusting Shield,http://eldenring.wiki.fextralife.com/file/Elde...,10.5,Silver thrusting shield embedded with glintsto...,1,"{'Str': 17, 'Dex': 13, 'Int': 15}",Standard/Pierce,Thrusting Shields,No passive effects,Shield Strike,0
2,2,3,Dryleaf Arts,http://eldenring.wiki.fextralife.com/file/Elde...,1.0,A technique for hand-to-hand combat without th...,1,"{'Str': 8, 'Dex': 8}",Strike,Hand-to-Hand Arts,No passive effects,Palm Blast,14
3,3,4,Dane's Footwork,http://eldenring.wiki.fextralife.com/file/Elde...,1.0,A technique for hand-to-hand combat without th...,1,"{'Str': 8, 'Dex': 8}",Strike,Hand-to-Hand Arts,No passive effects,Palm Blast,14
4,4,5,Smithscript Dagger,http://eldenring.wiki.fextralife.com/file/Elde...,1.5,Dagger engraved with a smithscript. Reduced ma...,1,"{'Str': 5, 'Dex': 11, 'Int': 11, 'Fai': 11}",Pierce,Throwing Blades,No passive effects,Piercing Throw,6


In [10]:
weapons_upgrades = pd.read_csv(BASE + 'weapons_upgrades.csv')
print("weapons_upgrades:", weapons_upgrades.shape)
print(weapons_upgrades.columns.tolist())
weapons_upgrades.head()

weapons_upgrades: (60201, 7)
['id', 'weapon name', 'upgrade', 'attack power', 'stat scaling', 'passive effects', 'damage reduction (%)']


,id,weapon name,upgrade,attack power,stat scaling,passive effects,damage reduction (%)
0,0,Dueling Shield,Standard,"{'Phy': '125 ', 'Mag': '- ', 'Fir': '- ', 'Lit...","{'Str': 'D ', 'Dex': 'D ', 'Int': '- ', 'Fai':...",{'Any': '- '},"{'Phy': '88 ', 'Mag': '44 ', 'Fir': '39 ', 'Li..."
1,1,Dueling Shield,Standard +1,"{'Phy': '132 ', 'Mag': '- ', 'Fir': '- ', 'Lit...","{'Str': 'D ', 'Dex': 'D ', 'Int': '- ', 'Fai':...",{'Any': '- '},"{'Phy': '88 ', 'Mag': '44 ', 'Fir': '39 ', 'Li..."
2,2,Dueling Shield,Standard +2,"{'Phy': '139 ', 'Mag': '- ', 'Fir': '- ', 'Lit...","{'Str': 'D ', 'Dex': 'D ', 'Int': '- ', 'Fai':...",{'Any': '- '},"{'Phy': '88 ', 'Mag': '44 ', 'Fir': '39 ', 'Li..."
3,3,Dueling Shield,Standard +3,"{'Phy': '146 ', 'Mag': '- ', 'Fir': '- ', 'Lit...","{'Str': 'D ', 'Dex': 'D ', 'Int': '- ', 'Fai':...",{'Any': '- '},"{'Phy': '88 ', 'Mag': '44 ', 'Fir': '39 ', 'Li..."
4,4,Dueling Shield,Standard +4,"{'Phy': '154 ', 'Mag': '- ', 'Fir': '- ', 'Lit...","{'Str': 'D ', 'Dex': 'D ', 'Int': '- ', 'Fai':...",{'Any': '- '},"{'Phy': '88 ', 'Mag': '44 ', 'Fir': '39 ', 'Li..."


In [11]:
ashes = pd.read_csv(BASE + 'ashesOfWar.csv')
print("ashesOfWar:", ashes.shape)
print(ashes.columns.tolist())
ashes.head()

ashesOfWar: (117, 7)
['id', 'name', 'image', 'affinity', 'skill', 'description', 'dlc']


,id,name,image,affinity,skill,description,dlc
0,0,Ash of War: Dryleaf Whirlwind,http://eldenring.wiki.fextralife.com/file/Elde...,Standard,Dryleaf Whirlwind,This Ash of War grants no affinity to an armam...,1
1,1,Ash of War: Aspects of the Crucible: Wings,http://eldenring.wiki.fextralife.com/file/Elde...,Sacred,Aspects of the Crucible: Wings,This Ash of War grants an armament the Sacred ...,1
2,2,Ash of War: Spinning Gravity Thrust,http://eldenring.wiki.fextralife.com/file/Elde...,Heavy,Spinning Gravity Thrust,This Ash of War grants an armament the Heavy a...,1
3,3,Ash of War: Palm Blast,http://eldenring.wiki.fextralife.com/file/Elde...,Standard,Palm Blast,This Ash of War grants no affinity to an armam...,1
4,4,Ash of War: Piercing Throw,http://eldenring.wiki.fextralife.com/file/Elde...,Keen,Piercing Throw,This Ash of War grants an armament the Keen af...,1


In [12]:
talismans = pd.read_csv(BASE + 'talismans.csv')
print("talismans:", talismans.shape)
print(talismans.columns.tolist())
talismans.head()

talismans: (155, 8)
['id', 'name', 'effect', 'weight', 'value', 'description', 'dlc', 'image']


,id,name,effect,weight,value,description,dlc,image
0,0,Crimson Amber Medallion,Effect Raises maximum HP by 6%,0.3,100,A medallion with crimson amber inlaid. Boosts ...,0,https://eldenring.wiki.fextralife.com//file/El...
1,1,Crimson Amber Medallion +1 Variant,Effect Raises maximum HP by 7%,0.3,500,A medallion with crimson amber inlaid. Boosts ...,0,https://eldenring.wiki.fextralife.com//file/El...
2,2,Crimson Amber Medallion +2 Variant,Effect Raises maximum HP by 8%,0.3,1000,A medallion with crimson amber inlaid. Boosts ...,0,https://eldenring.wiki.fextralife.com//file/El...
3,3,Crimson Amber Medallion +3 Variant,Effect Raises maximum HP by 10%,0.3,2000,A medallion with crimson amber inlaid. Boosts ...,1,https://eldenring.wiki.fextralife.com//file/El...
4,4,Cerulean Amber Medallion,Effect Raises maximum FP by 7,0.3,100,A medallion with cerulean amber inlaid.Boosts ...,0,https://eldenring.wiki.fextralife.com//file/El...


In [14]:
armors = pd.read_csv(BASE + 'armors.csv')
print("armors:", armors.shape)
print(armors.columns.tolist())
armors.head()

armors: (723, 12)
['id', 'name', 'image', 'description', 'type', 'damage negation', 'resistance', 'weight', 'special effect', 'how to acquire', 'in-game section', 'dlc']


,id,name,image,description,type,damage negation,resistance,weight,special effect,how to acquire,in-game section,dlc
0,0,Dane's Hat,http://eldenring.wiki.fextralife.com/file/Elde...,The sun-faded and lightly soiled hat of Drylea...,helm,"[{'Phy': '1.8', 'VS Str.': '2.3', 'VS Sla.': '...","[{'Imm.': '16', 'Rob.': '11', 'Foc.': '33', 'V...",2.2,NaN,This Helm is obtained as a drop upon defeating...,6,1
1,1,Gaius's Helm,http://eldenring.wiki.fextralife.com/file/Elde...,The black iron helm of Commander Gaius. Part o...,helm,"[{'Phy': '6.1', 'VS Str.': '5.6', 'VS Sla.': '...","[{'Imm.': '29', 'Rob.': '35', 'Foc.': '23', 'V...",8.6,NaN,Purchased from the Enia at the Roundtable Hold...,12,1
2,2,Oathseeker Knight Helm,http://eldenring.wiki.fextralife.com/file/Elde...,An Oathseeker Knight helm. Black with gold orn...,helm,"[{'Phy': '4.4', 'VS Str.': '3.8', 'VS Sla.': '...","[{'Imm.': '12', 'Rob.': '21', 'Foc.': '10', 'V...",4.4,NaN,"Found inside a hanging body, just outside the ...",2,1
3,3,Verdigris Helm,http://eldenring.wiki.fextralife.com/file/Elde...,Helm made from an unusual metal known as verdi...,helm,"[{'Phy': '7.5', 'VS Str.': '6.4', 'VS Sla.': '...","[{'Imm.': '39', 'Rob.': '46', 'Foc.': '15', 'V...",11.1,NaN,Complete Moore's questline,19,1
4,4,Pelt of Ralva,http://eldenring.wiki.fextralife.com/file/Elde...,"The pelt of Ralva the Great Red Bear, worn upo...",helm,"[{'Phy': '3.1', 'VS Str.': '3.4', 'VS Sla.': '...","[{'Imm.': '27', 'Rob.': '23', 'Foc.': '22', 'V...",3.6,"Enhances incantations of ""Bear Communion"" by 15%.",This is a helmet type of armor that can be obt...,6,1


In [15]:
incantations = pd.read_csv(BASE + 'incantations.csv')
sorceries = pd.read_csv(BASE + 'sorceries.csv')
print("incantations:", incantations.shape, incantations.columns.tolist())
print("sorceries:", sorceries.shape, sorceries.columns.tolist())

incantations: (129, 15) ['id', 'name', 'image', 'description', 'effect', 'FP', 'slot', 'INT', 'FAI', 'ARC', 'stamina cost', 'bonus', 'group', 'location', 'dlc']
sorceries: (84, 14) ['id', 'name', 'image', 'description', 'effect', 'FP', 'slot', 'INT', 'FAI', 'ARC', 'stamina cost', 'bonus', 'location', 'dlc']


What we can observe:

1. weapons_upgrades stores attack power and stat scaling as stringified dicts. We will need to parse these and not treat them as numbers.
2. requirements in weapons is also a stringified dict and will need to be handled the same way.
3. weapons_upgrades has 60,201 rows because every weapon has multiple upgrade tiers. The tool should be able to filter to the max upgrade.
4. Ashes of War has an affinity column which is important for build optimization as affinity changes damage scaling.

## Section 2: Data Cleaning

The core files store nested data as stringified Python dicts (attack power, stat scaling, requirements).
We need to parse these into usable structures before we can query them meaningfully in our agent tools.

In [16]:
# "The ast module helps Python applications to process trees of the Python abstract syntax grammar." [https://docs.python.org/3/library/ast.html]
import ast

def safe_parse(val):
    try:
        return ast.literal_eval(val)
    except:
        return {}

# Parse weapons requirements
weapons['requirements_parsed'] = weapons['requirements'].apply(safe_parse)

# Verify
weapons[['name', 'requirements', 'requirements_parsed']].head(3)

,name,requirements,requirements_parsed
0,Dueling Shield,"{'Str': 15, 'Dex': 14}","{'Str': 15, 'Dex': 14}"
1,Carian Thrusting Shield,"{'Str': 17, 'Dex': 13, 'Int': 15}","{'Str': 17, 'Dex': 13, 'Int': 15}"
2,Dryleaf Arts,"{'Str': 8, 'Dex': 8}","{'Str': 8, 'Dex': 8}"


In [17]:
# Parse weapons_upgrades attack power and stat scaling
weapons_upgrades['attack_power_parsed'] = weapons_upgrades['attack power'].apply(safe_parse)
weapons_upgrades['stat_scaling_parsed'] = weapons_upgrades['stat scaling'].apply(safe_parse)

weapons_upgrades[['weapon name', 'upgrade', 'attack_power_parsed', 'stat_scaling_parsed']].head(3)

,weapon name,upgrade,attack_power_parsed,stat_scaling_parsed
0,Dueling Shield,Standard,"{'Phy': '125 ', 'Mag': '- ', 'Fir': '- ', 'Lit...","{'Str': 'D ', 'Dex': 'D ', 'Int': '- ', 'Fai':..."
1,Dueling Shield,Standard +1,"{'Phy': '132 ', 'Mag': '- ', 'Fir': '- ', 'Lit...","{'Str': 'D ', 'Dex': 'D ', 'Int': '- ', 'Fai':..."
2,Dueling Shield,Standard +2,"{'Phy': '139 ', 'Mag': '- ', 'Fir': '- ', 'Lit...","{'Str': 'D ', 'Dex': 'D ', 'Int': '- ', 'Fai':..."


In [18]:
# Check what upgrade tiers exist — we need to know the max upgrade names
weapons_upgrades['upgrade'].unique()

array(['Standard ', 'Standard +1 ', 'Standard +2 ', 'Standard +3 ',
       'Standard +4 ', 'Standard +5 ', 'Standard +6 ', 'Standard +7 ',
       'Standard +8 ', 'Standard +9 ', 'Standard +10 ', 'Standard +11 ',
       'Standard +12 ', 'Standard +13 ', 'Standard +14 ', 'Standard +15 ',
       'Standard +16 ', 'Standard +17 ', 'Standard +18 ', 'Standard +19 ',
       'Standard +20 ', 'Standard +21 ', 'Standard +22 ', 'Standard +23 ',
       'Standard +24 ', 'Standard +25 ', 'Heavy ', 'Heavy +1 ',
       'Heavy +2 ', 'Heavy +3 ', 'Heavy +4 ', 'Heavy +5 ', 'Heavy +6 ',
       'Heavy +7 ', 'Heavy +8 ', 'Heavy +9 ', 'Heavy +10 ', 'Heavy +11 ',
       'Heavy +12 ', 'Heavy +13 ', 'Heavy +14 ', 'Heavy +15 ',
       'Heavy +16 ', 'Heavy +17 ', 'Heavy +18 ', 'Heavy +19 ',
       'Heavy +20 ', 'Heavy +21 ', 'Heavy +22 ', 'Heavy +23 ',
       'Heavy +24 ', 'Heavy +25 ', 'Keen ', 'Keen +1 ', 'Keen +2 ',
       'Keen +3 ', 'Keen +4 ', 'Keen +5 ', 'Keen +6 ', 'Keen +7 ',
       'Keen +8 ', 'Keen +9 '

Observations:
1. All upgrade tiers go to +25 max. There are no Somber weapons in this dataset (Somber goes to +10). That simplifies our max upgrade filter
2. The Flame affinity has \xa0 (non-breaking space) encoding issues: 'Flame\xa0+1 ' instead of 'Flame +1 '. Needs cleaning
3. Every upgrade string has a trailing space: 'Standard +25 ' not 'Standard +25'. Must strip before filtering
4. Stat scaling values are strings like 'D ' are not numeric. We'll need a scaling grade mapper (E < D < C < B < A < S) for the optimizer to reason about

## Section 3: Upgrade Data Cleaning

The upgrade column has trailing whitespace and encoding issues (non-breaking spaces in "Flame" affinity).
We also need a utility to identify max upgrade rows and convert letter scaling grades to numeric values
for comparison.

In [19]:
# Strip whitespace and fix non-breaking spaces in upgrade column
weapons_upgrades['upgrade'] = (
    weapons_upgrades['upgrade']
    .str.replace('\xa0', ' ', regex=False)
    .str.strip()
)

# Verify Flame is fixed
weapons_upgrades[weapons_upgrades['upgrade'].str.startswith('Flame')]['upgrade'].unique()

array(['Flame', 'Flame +1', 'Flame +2', 'Flame+3', 'Flame +4', 'Flame +5',
       'Flame +6', 'Flame +7', 'Flame +8', 'Flame +9', 'Flame +10',
       'Flame +11', 'Flame +12', 'Flame +13', 'Flame +14', 'Flame +15',
       'Flame +16', 'Flame +17', 'Flame +18', 'Flame +19', 'Flame +20',
       'Flame +21', 'Flame +22', 'Flame +23', 'Flame +24', 'Flame +25'],
      dtype=object)

In [20]:
# Define max upgrade per affinity — all standard affinities cap at +25
MAX_UPGRADES = [
    'Standard +25', 'Heavy +25', 'Keen +25', 'Quality +25',
    'Fire +25', 'Flame +25', 'Lightning +25', 'Sacred +25',
    'Magic +25', 'Cold +25', 'Poison +25', 'Blood +25', 'Occult +25'
]

# Filter to only max upgrade rows
weapons_max = weapons_upgrades[weapons_upgrades['upgrade'].isin(MAX_UPGRADES)].copy()
print(f"Max upgrade rows: {len(weapons_max)}")
weapons_max.head()

Max upgrade rows: 2238


,id,weapon name,upgrade,attack power,stat scaling,passive effects,damage reduction (%),attack_power_parsed,stat_scaling_parsed
25,25,Dueling Shield,Standard +25,"{'Phy': '306 ', 'Mag': '- ', 'Fir': '- ', 'Lit...","{'Str': 'D ', 'Dex': 'C ', 'Int': '- ', 'Fai':...",{'Any': '- '},"{'Phy': '88 ', 'Mag': '44 ', 'Fir': '39 ', 'Li...","{'Phy': '306 ', 'Mag': '- ', 'Fir': '- ', 'Lit...","{'Str': 'D ', 'Dex': 'C ', 'Int': '- ', 'Fai':..."
51,51,Dueling Shield,Heavy +25,"{'Phy': '284 ', 'Mag': '- ', 'Fir': '- ', 'Lit...","{'Str': 'B ', 'Dex': '- ', 'Int': '- ', 'Fai':...",{'Any': '- '},"{'Phy': '88 ', 'Mag': '44 ', 'Fir': '39 ', 'Li...","{'Phy': '284 ', 'Mag': '- ', 'Fir': '- ', 'Lit...","{'Str': 'B ', 'Dex': '- ', 'Int': '- ', 'Fai':..."
77,77,Dueling Shield,Keen +25,"{'Phy': '284 ', 'Mag': '- ', 'Fir': '- ', 'Lit...","{'Str': 'E ', 'Dex': 'B ', 'Int': '- ', 'Fai':...",{'Any': '- '},"{'Phy': '88 ', 'Mag': '44 ', 'Fir': '39 ', 'Li...","{'Phy': '284 ', 'Mag': '- ', 'Fir': '- ', 'Lit...","{'Str': 'E ', 'Dex': 'B ', 'Int': '- ', 'Fai':..."
103,103,Dueling Shield,Quality +25,"{'Phy': '248 ', 'Mag': '- ', 'Fir': '- ', 'Lit...","{'Str': 'B ', 'Dex': 'B ', 'Int': '- ', 'Fai':...",{'Any': '- '},"{'Phy': '88 ', 'Mag': '44 ', 'Fir': '39 ', 'Li...","{'Phy': '248 ', 'Mag': '- ', 'Fir': '- ', 'Lit...","{'Str': 'B ', 'Dex': 'B ', 'Int': '- ', 'Fai':..."
129,129,Dueling Shield,Fire +25,"{'Phy': '205 ', 'Mag': '- ', 'Fir': '205 ', 'L...","{'Str': 'C ', 'Dex': 'E ', 'Int': '- ', 'Fai':...",{'Any': '- '},"{'Phy': '83.6 ', 'Mag': '41.8 ', 'Fir': '48.75...","{'Phy': '205 ', 'Mag': '- ', 'Fir': '205 ', 'L...","{'Str': 'C ', 'Dex': 'E ', 'Int': '- ', 'Fai':..."


In [21]:
# Scaling grade to numeric mapper — needed so agent can reason "A scaling > C scaling"
SCALING_MAP = {'S': 6, 'A': 5, 'B': 4, 'C': 3, 'D': 2, 'E': 1, '-': 0}

def parse_scaling_grade(val):
    if isinstance(val, str):
        return SCALING_MAP.get(val.strip(), 0)
    return 0

# Apply to stat_scaling_parsed — extract each stat's numeric grade
def expand_scaling(row):
    parsed = row['stat_scaling_parsed']
    if not isinstance(parsed, dict):
        return pd.Series({'Str_scale': 0, 'Dex_scale': 0, 'Int_scale': 0, 'Fai_scale': 0, 'Arc_scale': 0})
    return pd.Series({
        'Str_scale': parse_scaling_grade(parsed.get('Str', '-')),
        'Dex_scale': parse_scaling_grade(parsed.get('Dex', '-')),
        'Int_scale': parse_scaling_grade(parsed.get('Int', '-')),
        'Fai_scale': parse_scaling_grade(parsed.get('Fai', '-')),
        'Arc_scale': parse_scaling_grade(parsed.get('Arc', '-')),
    })

weapons_max = pd.concat([weapons_max, weapons_max.apply(expand_scaling, axis=1)], axis=1)
weapons_max[['weapon name', 'upgrade', 'Str_scale', 'Dex_scale', 'Int_scale', 'Fai_scale', 'Arc_scale']].head(10)

,weapon name,upgrade,Str_scale,Dex_scale,Int_scale,Fai_scale,Arc_scale
25,Dueling Shield,Standard +25,2,3,0,0,0
51,Dueling Shield,Heavy +25,4,0,0,0,0
77,Dueling Shield,Keen +25,1,4,0,0,0
103,Dueling Shield,Quality +25,4,4,0,0,0
129,Dueling Shield,Fire +25,3,1,0,0,0
155,Dueling Shield,Flame +25,1,2,0,3,0
181,Dueling Shield,Lightning +25,1,3,0,0,0
207,Dueling Shield,Sacred +25,1,2,0,3,0
233,Dueling Shield,Magic +25,1,1,4,0,0
259,Dueling Shield,Cold +25,2,3,3,0,0


In [22]:
# Also parse physical attack power to numeric for sorting
def get_phys_attack(row):
    parsed = row['attack_power_parsed']
    if not isinstance(parsed, dict):
        return 0
    try:
        return float(str(parsed.get('Phy', '0')).strip().replace('-', '0') or 0)
    except:
        return 0

weapons_max['phys_attack'] = weapons_max.apply(get_phys_attack, axis=1)
weapons_max[['weapon name', 'upgrade', 'phys_attack', 'Str_scale', 'Fai_scale']].sort_values('phys_attack', ascending=False).head(10)

,weapon name,upgrade,phys_attack,Str_scale,Fai_scale
59363,Hand Ballista,Standard +25,600.0,0,0
42349,Duelist Greataxe,Standard +25,416.0,2,0
13836,Greatsword,Standard +25,401.0,4,0
42687,Rotten Greataxe,Standard +25,396.0,2,0
43025,Golem's Halberd,Standard +25,387.0,3,0
42375,Duelist Greataxe,Heavy +25,385.0,4,0
43701,Prelate's Inferno Crozier,Standard +25,382.0,4,0
14512,Troll's Golden Sword,Standard +25,379.0,3,0
43363,Giant-Crusher,Standard +25,379.0,4,0
44039,Great Club,Standard +25,377.0,4,2


From this we learn that:
1. Flame+3 (no space) is still a one-off typo in the data and Flame +25 is fine so our max filter works correctly
2. 2,238 max upgrade rows across all affinities which is a very manageable size for tool queries
3. Hand Ballista at 600 physical is an outlier (it's a crossbow, raw damage doesn't scale the same way)- will note this and research the weapon but it's probably not a problem
4. The scaling numeric system is working correctly. For example we can see Greatsword at Str 4 (B scaling), Faith 0

## Section 4: Tool Definitions

Here we define the functions our AI agent can call. Each tool wraps a pandas query
over our cleaned dataframes. The agent will decide which tools to call and in what
order based on the user's build request.

Tools:
- `query_weapons_by_stat` — filter weapons by stat requirements and scaling preference
- `get_weapon_upgrades` — get all affinity options at max upgrade for a specific weapon
- `query_talismans` — search talismans by keyword or effect
- `query_ashes_of_war` — find ashes of war by affinity or skill name
- `query_spells` — search incantations or sorceries by stat requirement

In [23]:
def query_weapons_by_stat(
    stat: str = None,
    min_scaling_grade: int = 0,
    category: str = None,
    max_str_req: int = 99,
    max_dex_req: int = 99,
    max_int_req: int = 99,
    max_fai_req: int = 99,
    max_arc_req: int = 99,
    affinity: str = None,
    top_n: int = 10
) -> str:
    """
    Query weapons_max for weapons matching stat scaling and requirement filters.
    stat: which stat to sort by ('Str', 'Dex', 'Int', 'Fai', 'Arc')
    min_scaling_grade: minimum numeric grade (0-6) for that stat
    affinity: filter to a specific affinity e.g. 'Sacred', 'Heavy'
    Returns top_n results as JSON string.
    """
    df = weapons_max.copy()

    # Merge with weapons to get requirements and category
    df = df.merge(
        weapons[['name', 'requirements_parsed', 'category']],
        left_on='weapon name', right_on='name', how='left'
    )

    # Filter by stat requirements
    def meets_reqs(req_dict):
        if not isinstance(req_dict, dict):
            return True
        return (
            req_dict.get('Str', 0) <= max_str_req and
            req_dict.get('Dex', 0) <= max_dex_req and
            req_dict.get('Int', 0) <= max_int_req and
            req_dict.get('Fai', 0) <= max_fai_req and
            req_dict.get('Arc', 0) <= max_arc_req
        )

    df = df[df['requirements_parsed'].apply(meets_reqs)]

    # Filter by affinity
    if affinity:
        df = df[df['upgrade'].str.startswith(affinity)]

    # Filter by category
    if category:
        df = df[df['category'].str.contains(category, case=False, na=False)]

    # Filter by scaling grade
    scale_col_map = {
        'Str': 'Str_scale', 'Dex': 'Dex_scale',
        'Int': 'Int_scale', 'Fai': 'Fai_scale', 'Arc': 'Arc_scale'
    }
    if stat and stat in scale_col_map:
        col = scale_col_map[stat]
        df = df[df[col] >= min_scaling_grade]
        df = df.sort_values(col, ascending=False)
    else:
        df = df.sort_values('phys_attack', ascending=False)

    cols = ['weapon name', 'upgrade', 'phys_attack', 'Str_scale',
            'Dex_scale', 'Int_scale', 'Fai_scale', 'Arc_scale', 'category']
    return df[cols].head(top_n).to_json(orient='records')

In [24]:
def get_weapon_upgrades(weapon_name: str) -> str:
    """
    Get all max-upgrade affinity options for a specific weapon.
    Returns scaling and attack power across all affinities.
    """
    df = weapons_max[
        weapons_max['weapon name'].str.lower() == weapon_name.lower()
    ].copy()

    if df.empty:
        # Try partial match
        df = weapons_max[
            weapons_max['weapon name'].str.lower().str.contains(weapon_name.lower())
        ].copy()

    cols = ['weapon name', 'upgrade', 'phys_attack',
            'Str_scale', 'Dex_scale', 'Int_scale', 'Fai_scale', 'Arc_scale']
    return df[cols].to_json(orient='records')

In [25]:
def query_talismans(keyword: str = None, top_n: int = 10) -> str:
    """
    Search talismans by keyword in name or effect column.
    """
    df = talismans.copy()
    if keyword:
        mask = (
            df['name'].str.contains(keyword, case=False, na=False) |
            df['effect'].str.contains(keyword, case=False, na=False) |
            df['description'].str.contains(keyword, case=False, na=False)
        )
        df = df[mask]
    return df[['name', 'effect', 'weight', 'dlc']].head(top_n).to_json(orient='records')

In [26]:
def query_ashes_of_war(affinity: str = None, skill_keyword: str = None, top_n: int = 10) -> str:
    """
    Search ashes of war by affinity type or skill name keyword.
    """
    df = ashes.copy()
    if affinity:
        df = df[df['affinity'].str.contains(affinity, case=False, na=False)]
    if skill_keyword:
        df = df[df['skill'].str.contains(skill_keyword, case=False, na=False)]
    return df[['name', 'affinity', 'skill', 'dlc']].head(top_n).to_json(orient='records')

In [27]:
def query_spells(
    spell_type: str = 'incantation',
    max_fai: int = 99,
    max_int: int = 99,
    max_arc: int = 99,
    keyword: str = None,
    top_n: int = 10
) -> str:
    """
    Query incantations or sorceries by stat requirements.
    spell_type: 'incantation' or 'sorcery'
    """
    df = incantations.copy() if spell_type == 'incantation' else sorceries.copy()

    df = df[
        (pd.to_numeric(df['FAI'], errors='coerce').fillna(0) <= max_fai) &
        (pd.to_numeric(df['INT'], errors='coerce').fillna(0) <= max_int) &
        (pd.to_numeric(df['ARC'], errors='coerce').fillna(0) <= max_arc)
    ]

    if keyword:
        df = df[
            df['name'].str.contains(keyword, case=False, na=False) |
            df['effect'].str.contains(keyword, case=False, na=False)
        ]

    cols = ['name', 'FP', 'FAI', 'INT', 'ARC', 'effect', 'dlc']
    return df[cols].head(top_n).to_json(orient='records')

## Section 5: Agent Tool Schema

Here we define the tool schemas in Gemini's function calling format. These tell the
model what tools it has access to, what parameters each accepts, and when to use them.
The model will decide which tools to call and in what order based on the user's request.

In [ ]:
from google import genai
from google.genai import types

API_KEY = ""
client = genai.Client(api_key=API_KEY)

# Quick sanity check
# response = client.models.generate_content(
#     model="gemini-2.0-flash",
#     contents="Say 'Elden Ring agent ready' and nothing else."
# )
response = client.models.generate_content(
    model="models/gemini-flash-latest",
    contents="Say 'Elden Ring agent ready' and nothing else."
)
print(response.text)

Elden Ring agent ready


In [29]:
from google.genai import types

gemini_tools = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="query_weapons_by_stat",
            description=(
                "Search for weapons filtered by stat scaling and requirements. "
                "Use when the user specifies a primary stat (Str, Dex, Int, Fai, Arc), "
                "a weapon category, or stat caps. Returns top weapons with scaling grades and attack power."
            ),
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "stat": types.Schema(type=types.Type.STRING, description="Primary stat to sort by: 'Str', 'Dex', 'Int', 'Fai', or 'Arc'"),
                    "min_scaling_grade": types.Schema(type=types.Type.INTEGER, description="Minimum scaling grade (0=none, 1=E, 2=D, 3=C, 4=B, 5=A, 6=S)"),
                    "category": types.Schema(type=types.Type.STRING, description="Weapon category filter e.g. 'Greatsword', 'Dagger', 'Colossal'"),
                    "max_str_req": types.Schema(type=types.Type.INTEGER, description="Maximum Strength requirement"),
                    "max_dex_req": types.Schema(type=types.Type.INTEGER, description="Maximum Dexterity requirement"),
                    "max_int_req": types.Schema(type=types.Type.INTEGER, description="Maximum Intelligence requirement"),
                    "max_fai_req": types.Schema(type=types.Type.INTEGER, description="Maximum Faith requirement"),
                    "max_arc_req": types.Schema(type=types.Type.INTEGER, description="Maximum Arcane requirement"),
                    "affinity": types.Schema(type=types.Type.STRING, description="Affinity filter e.g. 'Sacred', 'Heavy', 'Blood', 'Occult'"),
                    "top_n": types.Schema(type=types.Type.INTEGER, description="Number of results to return (default 10)"),
                }
            )
        ),
        types.FunctionDeclaration(
            name="get_weapon_upgrades",
            description=(
                "Get all affinity options at max upgrade level for a specific weapon. "
                "Use this to compare how a weapon performs across different affinities "
                "e.g. Heavy vs Sacred vs Occult for a specific weapon name."
            ),
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "weapon_name": types.Schema(type=types.Type.STRING, description="The name of the weapon to look up e.g. 'Greatsword', 'Rivers of Blood'"),
                },
                required=["weapon_name"]
            )
        ),
        types.FunctionDeclaration(
            name="query_talismans",
            description=(
                "Search for talismans by keyword in name, effect, or description. "
                "Use this to find talismans that complement a build e.g. 'faith', 'strength', "
                "'stamina', 'FP', 'charged attack'."
            ),
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "keyword": types.Schema(type=types.Type.STRING, description="Keyword to search in talisman name, effect, or description"),
                    "top_n": types.Schema(type=types.Type.INTEGER, description="Number of results to return (default 10)"),
                }
            )
        ),
        types.FunctionDeclaration(
            name="query_ashes_of_war",
            description=(
                "Search for Ashes of War by affinity or skill name keyword. "
                "Use this to find weapon skills that match the build's affinity or playstyle."
            ),
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "affinity": types.Schema(type=types.Type.STRING, description="Affinity type e.g. 'Sacred', 'Heavy', 'Blood', 'Occult'"),
                    "skill_keyword": types.Schema(type=types.Type.STRING, description="Keyword to search in skill name e.g. 'Storm', 'Flame', 'Gravity'"),
                    "top_n": types.Schema(type=types.Type.INTEGER, description="Number of results to return (default 10)"),
                }
            )
        ),
        types.FunctionDeclaration(
            name="query_spells",
            description=(
                "Search incantations or sorceries by stat requirements and keyword. "
                "Use this for spellcaster or hybrid builds to find usable spells."
            ),
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "spell_type": types.Schema(type=types.Type.STRING, description="'incantation' for faith/arc spells, 'sorcery' for int spells"),
                    "max_fai": types.Schema(type=types.Type.INTEGER, description="Maximum Faith requirement"),
                    "max_int": types.Schema(type=types.Type.INTEGER, description="Maximum Intelligence requirement"),
                    "max_arc": types.Schema(type=types.Type.INTEGER, description="Maximum Arcane requirement"),
                    "keyword": types.Schema(type=types.Type.STRING, description="Keyword to search in spell name or effect"),
                    "top_n": types.Schema(type=types.Type.INTEGER, description="Number of results to return (default 10)"),
                }
            )
        ),
    ]
)

print(f"{len(gemini_tools.function_declarations)} tools defined")

5 tools defined


## Section 6: Tool Dispatcher

Routes the model's function call to the correct Python function and returns the result
back into the conversation as a tool response.

In [30]:
def dispatch_tool(tool_name: str, tool_input: dict) -> str:
    if tool_name == "query_weapons_by_stat":
        return query_weapons_by_stat(**tool_input)
    elif tool_name == "get_weapon_upgrades":
        return get_weapon_upgrades(**tool_input)
    elif tool_name == "query_talismans":
        return query_talismans(**tool_input)
    elif tool_name == "query_ashes_of_war":
        return query_ashes_of_war(**tool_input)
    elif tool_name == "query_spells":
        return query_spells(**tool_input)
    else:
        return json.dumps({"error": f"Unknown tool: {tool_name}"})

print("Dispatcher ready")

Dispatcher ready


## Section 7: Agent Loop

The agent loop is the heart of the build optimizer. It sends the user's request to
Gemini, checks if the model wants to call a tool, executes it, feeds the result back,
and repeats until the model has enough information to give a final answer.
This loop is what makes this an AI agent rather than a simple chatbot.

In [31]:
def run_build_agent(user_request: str, max_turns: int = 10, verbose: bool = True) -> str:
    """
    Run the Elden Ring build optimizer agent.
    
    user_request: natural language build request from the user
    max_turns: maximum tool call iterations before forcing a final answer
    verbose: if True, prints each tool call and result as it happens
    """
    
    system_prompt = """You are an Elden Ring build optimizer. You have access to tools 
that query a real dataset of weapons, upgrades, talismans, ashes of war, and spells.

When a user describes a build goal, you MUST use the tools to look up real data before 
making recommendations. Never recommend weapons or talismans from memory alone.

For every build request:
1. Query weapons matching the user's stat spread and playstyle
2. Check upgrade affinities for the top weapon candidates
3. Find complementary talismans
4. Find relevant ashes of war or spells if applicable
5. Synthesize a complete build recommendation with data-backed reasoning

Always explain WHY you are recommending each item based on what the data shows."""

    messages = [
        types.Content(
            role="user",
            parts=[types.Part(text=user_request)]
        )
    ]
    
    turn = 0
    
    while turn < max_turns:
        turn += 1
        
        if verbose:
            print(f"\n--- Turn {turn} ---")
        
        # Send to Gemini
        response = client.models.generate_content(
            model="models/gemini-flash-latest",
            contents=messages,
            config=types.GenerateContentConfig(
                system_instruction=system_prompt,
                tools=[gemini_tools],
                temperature=0.2
            )
        )
        
        candidate = response.candidates[0]
        
        # Add model response to message history
        messages.append(
            types.Content(role="model", parts=candidate.content.parts)
        )
        
        # Check if model wants to call tools
        tool_calls = [p for p in candidate.content.parts if p.function_call is not None]
        
        if not tool_calls:
            # No tool calls — model is giving final answer
            final_text = " ".join(
                p.text for p in candidate.content.parts if p.text
            )
            if verbose:
                print("\n=== FINAL BUILD RECOMMENDATION ===")
                print(final_text)
            return final_text
        
        # Execute each tool call and collect results
        tool_response_parts = []
        
        for part in tool_calls:
            fn = part.function_call
            tool_name = fn.name
            tool_input = dict(fn.args)
            
            if verbose:
                print(f"Tool call: {tool_name}({tool_input})")
            
            result = dispatch_tool(tool_name, tool_input)
            
            if verbose:
                # Print a preview of the result, not the whole thing
                preview = result[:300] + "..." if len(result) > 300 else result
                print(f"Result preview: {preview}")
            
            tool_response_parts.append(
                types.Part(
                    function_response=types.FunctionResponse(
                        name=tool_name,
                        response={"result": result}
                    )
                )
            )
        
        # Feed tool results back into conversation
        messages.append(
            types.Content(role="user", parts=tool_response_parts)
        )
    
    return "Max turns reached without final answer."

print("Agent loop ready")

Agent loop ready


## Section 8: Test the Agent

In [32]:
result = run_build_agent(
    "I'm building a faith/strength character with 60 faith and 40 strength. "
    "I want a colossal weapon or greatsword. Recommend me a complete build."
)


--- Turn 1 ---
Tool call: query_weapons_by_stat({'max_fai_req': 60, 'category': 'Greatsword', 'max_str_req': 40, 'min_scaling_grade': 4, 'stat': 'Fai'})
Result preview: [{"weapon name":"Gargoyle's Greatsword","upgrade":"Flame +25","phys_attack":258.0,"Str_scale":2,"Dex_scale":1,"Int_scale":0,"Fai_scale":4,"Arc_scale":0,"category":"Greatswords"},{"weapon name":"Gargoyle's Greatsword","upgrade":"Sacred +25","phys_attack":258.0,"Str_scale":2,"Dex_scale":1,"Int_scale":...
Tool call: query_weapons_by_stat({'max_str_req': 40, 'max_fai_req': 60, 'category': 'Colossal Sword', 'stat': 'Fai', 'min_scaling_grade': 4})
Result preview: [{"weapon name":"Watchdog's Greatsword","upgrade":"Flame +25","phys_attack":274.0,"Str_scale":3,"Dex_scale":1,"Int_scale":0,"Fai_scale":4,"Arc_scale":0,"category":"Colossal Swords"},{"weapon name":"Watchdog's Greatsword","upgrade":"Sacred +25","phys_attack":274.0,"Str_scale":3,"Dex_scale":1,"Int_sca...
Tool call: query_weapons_by_stat({'max_str_req': 40, 'max_fai_re

From this we can see that the agent is working!
For each turn we observed:
- Turn 1: Queried weapons by category and faith scaling and found real candidates from the data
- Turn 2: Tried to look up specific unique weapons by name and got empty results (those weapons are likely in a different category in the dataset, worth noting)
- Turn 3: Queried talismans, spells, and ashes of war to flesh out the build
- Turn 4: Dug deeper on seals, specific talismans, and key incantations
- Turn 5: Synthesized everything into a full recommendation

## Section 9: Known Limitations

- **Unique/Somber weapons** (e.g. Blasphemous Blade, Maliketh's Black Blade) use a separate
  upgrade path (Somber Smithing Stones, +1 to +10) and are not present in `weapons_upgrades.csv`.
  The agent will return empty results when queried by name for these weapons.
  A future improvement we can do for the next section would be to add a separate somber weapons lookup tool and expand the agent.

## Section 10: Expanding the Agent

We expand the agent with additional tools covering spirit ashes, armor, skills, and item
locations. We also fix the somber weapons gap by investigating why unique weapons like
Blasphemous Blade are missing from our current dataset.

In [33]:
# First let's understand the somber weapons gap
# Check if these weapons exist in weapons.csv at all
somber_check = ['Blasphemous Blade', "Maliketh's Black Blade", "Ordovis's Greatsword", 
                'Rivers of Blood', 'Moonveil', 'Eleonora']

for name in somber_check:
    match = weapons[weapons['name'].str.contains(name, case=False, na=False)]
    print(f"{name}: {len(match)} rows in weapons.csv")
    if len(match) > 0:
        print(f"  -> weapon_id: {match['weapon_id'].values}")

Blasphemous Blade: 1 rows in weapons.csv
  -> weapon_id: [81]
Maliketh's Black Blade: 1 rows in weapons.csv
  -> weapon_id: [98]
Ordovis's Greatsword: 1 rows in weapons.csv
  -> weapon_id: [77]
Rivers of Blood: 1 rows in weapons.csv
  -> weapon_id: [154]
Moonveil: 1 rows in weapons.csv
  -> weapon_id: [153]
Eleonora: 1 rows in weapons.csv
  -> weapon_id: [164]


In [34]:
# Check if they appear in weapons_upgrades at all (before our max filter)
for name in somber_check:
    match = weapons_upgrades[weapons_upgrades['weapon name'].str.contains(name, case=False, na=False)]
    print(f"{name}: {len(match)} rows in weapons_upgrades.csv")
    if len(match) > 0:
        print(f"  -> upgrade tiers found: {match['upgrade'].unique()}")

Blasphemous Blade: 11 rows in weapons_upgrades.csv
  -> upgrade tiers found: ['Standard' 'Standard +1' 'Standard +2' 'Standard +3' 'Standard +4'
 'Standard +5' 'Standard +6' 'Standard +7' 'Standard +8' 'Standard +9'
 'Standard +10']
Maliketh's Black Blade: 22 rows in weapons_upgrades.csv
  -> upgrade tiers found: ['Standard' 'Standard +1' 'Standard +2' 'Standard +3' 'Standard +4'
 'Standard +5' 'Standard +6' 'Standard +7' 'Standard +8' 'Standard +9'
 'Standard +10']
Ordovis's Greatsword: 11 rows in weapons_upgrades.csv
  -> upgrade tiers found: ['Standard' 'Standard +1' 'Standard +2' 'Standard +3' 'Standard +4'
 'Standard +5' 'Standard +6' 'Standard +7' 'Standard +8' 'Standard +9'
 'Standard +10']
Rivers of Blood: 11 rows in weapons_upgrades.csv
  -> upgrade tiers found: ['Standard' 'Standard +1' 'Standard +2' 'Standard +3' 'Standard +4'
 'Standard +5' 'Standard +6' 'Standard +7' 'Standard +8' 'Standard +9'
 'Standard +10']
Moonveil: 11 rows in weapons_upgrades.csv
  -> upgrade tiers f

In [35]:
# Standard +10 = somber weapons (mislabeled in dataset as Standard instead of Somber)
# Standard +25 = regular weapons
# Both are valid max upgrade tiers

MAX_UPGRADES_UPDATED = MAX_UPGRADES + ['Standard +10']

weapons_max = weapons_upgrades[
    weapons_upgrades['upgrade'].isin(MAX_UPGRADES_UPDATED)
].copy()

print(f"Previous max upgrade rows: 2238")
print(f"Updated max upgrade rows: {len(weapons_max)}")

# Verify somber weapons are now included
for name in somber_check:
    match = weapons_max[weapons_max['weapon name'].str.contains(name, case=False, na=False)]
    print(f"{name}: {len(match)} rows at max upgrade")

Previous max upgrade rows: 2238
Updated max upgrade rows: 2643
Blasphemous Blade: 1 rows at max upgrade
Maliketh's Black Blade: 2 rows at max upgrade
Ordovis's Greatsword: 1 rows at max upgrade
Rivers of Blood: 1 rows at max upgrade
Moonveil: 1 rows at max upgrade
Eleonora: 1 rows at max upgrade


In [36]:
# Re-apply the scaling and attack power parsing to the updated weapons_max
weapons_max = pd.concat([weapons_max, weapons_max.apply(expand_scaling, axis=1)], axis=1)
weapons_max['phys_attack'] = weapons_max.apply(get_phys_attack, axis=1)

# Merge with weapons to restore requirements and category columns
weapons_max = weapons_max.merge(
    weapons[['name', 'requirements_parsed', 'category']],
    left_on='weapon name', right_on='name', how='left'
)

print("weapons_max rebuilt with somber weapons included")
weapons_max[weapons_max['weapon name'].isin(somber_check)][
    ['weapon name', 'upgrade', 'phys_attack', 'Str_scale', 'Dex_scale', 'Fai_scale', 'Arc_scale']
]

weapons_max rebuilt with somber weapons included


,weapon name,upgrade,phys_attack,Str_scale,Dex_scale,Fai_scale,Arc_scale
563,Ordovis's Greatsword,Standard +10,262.0,5,1,3,0
567,Blasphemous Blade,Standard +10,296.0,3,3,4,0
649,Maliketh's Black Blade,Standard +10,311.0,4,2,4,0
668,Maliketh's Black Blade,Standard +10,311.0,4,2,4,0
1083,Moonveil,Standard +10,178.0,1,4,0,0
1084,Rivers of Blood,Standard +10,186.0,1,4,0,2


Somber weapons are now in. A couple of things to note from this output:
Maliketh's Black Blade showing 2 rows is because it has two entries in the dataset. One is likely the two-handed variant. This is not a problem as the agent will just see both options.
Blasphemous Blade correctly shows Str 3 (C), Fai 4 (B) scaling which matches the real game. The fix worked.

## Section 11: Loading Additional Data Sources

In [37]:
# Load remaining files
spirit_ashes = pd.read_csv(BASE + 'spiritAshes.csv')
print("spiritAshes:", spirit_ashes.shape)
print(spirit_ashes.columns.tolist())
spirit_ashes.head(3)

spiritAshes: (84, 9)
['id', 'name', 'image', 'type', 'FP cost', 'HP cost', 'effect', 'description', 'dlc']


,id,name,image,type,FP cost,HP cost,effect,description,dlc
0,0,Curseblade Meera,http://eldenring.wiki.fextralife.com/file/Elde...,Spirit Summon,91,0,Summons spirit of Curseblade Meera,Ashen remains in which spirits yet dwell. Use ...,1
1,1,Bloodfiend Hexer's Ashes,http://eldenring.wiki.fextralife.com/file/Elde...,Spirit Summon,0,500,Summons bloodfiend hexer spirit,Ashen remains in which spirits yet dwell. Use ...,1
2,2,Gravebird Ashes,http://eldenring.wiki.fextralife.com/file/Elde...,Spirit Summon,52,0,Summons Gravebird spirit,Ashen remains in which spirits yet dwell. Use ...,1


In [38]:
skills = pd.read_csv(BASE + 'skills.csv')
print("skills:", skills.shape)
print(skills.columns.tolist())
skills.head(3)

skills: (257, 10)
['id', 'name', 'image', 'type', 'equipament', 'charge', 'FP', 'effect', 'locations', 'dlc']


,id,name,image,type,equipament,charge,FP,effect,locations,dlc
0,0,Dryleaf Whirlwind,http://eldenring.wiki.fextralife.com/file/Elde...,Regular,Usable on Hand-to-Hand Arts,No,9 (-/-),This skill represents the pinnacle of Dane's f...,Ash of War Dryleaf Whirlwind,1
1,1,Aspects of the Crucible: Wings,No Image,Regular,All Swords and Polearms capable of Thrusting (...,No,22 (-/-),This skill originates from the lifeforms of th...,Ash of War: Aspects of the Crucible: Wings,1
2,2,Spinning Gravity Thrust,http://eldenring.wiki.fextralife.com/file/Elde...,Regular,Usable on large and colossal swords,No,26 (-/12),A gravity skill honed by the disciples of an A...,Ash of War Spinning Gravity Thrust,1


In [39]:
locations = pd.read_csv(BASE + 'locations.csv')
print("locations:", locations.shape)
print(locations.columns.tolist())
locations.head(3)

locations: (286, 10)
['id', 'name', 'image', 'region', 'items', 'npcs', 'creatures', 'bosses', 'description', 'dlc']


,id,name,image,region,items,npcs,creatures,bosses,description,dlc
0,0,Abandoned Ailing Village,http://eldenring.wiki.fextralife.com/file/Elde...,Gravesite Plain,"['Broken Rune', 'Golden Rune (1)', 'Black Pyre...",['Spirit NPC'],['Man-Fly'],NaN,A dilapidated village found north of Ellac bri...,1
1,1,Artist's Shack (Gravesite Plain),http://eldenring.wiki.fextralife.com/file/Elde...,Gravesite Plain,['Incursion Painting'],NaN,NaN,NaN,A small ruined shack overlooking the Gravesite...,1
2,2,Belurat Gaol,http://eldenring.wiki.fextralife.com/file/Elde...,Gravesite Plain,"['Frozen Maggot', 'Broken Rune', 'Shadow Realm...",NaN,"['Living Jar', 'Jar Innards', 'Shadow Undead',...",['Demi-Human Swordmaster Onze'],An underground prison located deep within the ...,1


In [40]:
shields = pd.read_csv(BASE + 'shields.csv')
shields_upgrades = pd.read_csv(BASE + 'shields_upgrades.csv')
print("shields:", shields.shape, shields.columns.tolist())
print("shields_upgrades:", shields_upgrades.shape, shields_upgrades.columns.tolist())
shields.head(3)

shields: (100, 13) ['id', 'shield_id', 'name', 'image', 'weight', 'description', 'dlc', 'requirements', 'damage type', 'category', 'passive effect', 'skill', 'FP cost']
shields_upgrades: (28286, 7) ['id', 'shield name', 'upgrade', 'attack power', 'stat scaling', 'passive effects', 'damage reduction (%)']


,id,shield_id,name,image,weight,description,dlc,requirements,damage type,category,passive effect,skill,FP cost
0,0,1,Rickety Shield,http://eldenring.wiki.fextralife.com/file/Elde...,1.0,"A creaky old wooden shield, circular in shape....",0,{'Str': 8},Strike,Small Shields,No passive effects,Parry,0.0
1,1,2,Riveted Wooden Shield,http://eldenring.wiki.fextralife.com/file/Elde...,2.0,Small wooden roundshield that has been reinfor...,0,{'Str': 8},Strike,Small Shields,No passive effects,Parry,0.0
2,2,3,Riveted Wooden Shield,http://eldenring.wiki.fextralife.com/file/Elde...,2.0,Small wooden roundshield that has been reinfor...,0,{'Str': 8},Strike,Small Shields,No passive effects,Parry,0.0


## Section 12: New Tool Definitions

Adding five new tools to expand the agent's coverage: spirit ashes, armor, skills,
shields, and item locations. These allow the agent to recommend complete builds
including summons, defensive gear, and where to find everything.

In [41]:
def query_spirit_ashes(keyword: str = None, max_fp: int = 999, max_hp: int = 999, top_n: int = 10) -> str:
    """
    Search spirit ashes by keyword in name or effect, with FP/HP cost filters.
    Use this to recommend summons that complement the build.
    """
    df = spirit_ashes.copy()
    
    df['FP cost'] = pd.to_numeric(df['FP cost'], errors='coerce').fillna(0)
    df['HP cost'] = pd.to_numeric(df['HP cost'], errors='coerce').fillna(0)
    
    df = df[(df['FP cost'] <= max_fp) & (df['HP cost'] <= max_hp)]
    
    if keyword:
        mask = (
            df['name'].str.contains(keyword, case=False, na=False) |
            df['effect'].str.contains(keyword, case=False, na=False) |
            df['description'].str.contains(keyword, case=False, na=False)
        )
        df = df[mask]
    
    return df[['name', 'type', 'FP cost', 'HP cost', 'effect', 'dlc']].head(top_n).to_json(orient='records')

In [42]:
def query_armor(
    armor_type: str = None,
    keyword: str = None,
    max_weight: float = 99.0,
    top_n: int = 10
) -> str:
    """
    Search armor by type (helm, chest, gauntlets, leg), weight limit, or keyword
    in name, special effect, or description. Use to recommend armor for a build.
    """
    df = armors.copy()
    df['weight'] = pd.to_numeric(df['weight'], errors='coerce').fillna(0)
    df = df[df['weight'] <= max_weight]
    
    if armor_type:
        df = df[df['type'].str.contains(armor_type, case=False, na=False)]
    
    if keyword:
        mask = (
            df['name'].str.contains(keyword, case=False, na=False) |
            df['special effect'].fillna('').str.contains(keyword, case=False, na=False) |
            df['description'].str.contains(keyword, case=False, na=False)
        )
        df = df[mask]
    
    return df[['name', 'type', 'weight', 'special effect', 'dlc']].head(top_n).to_json(orient='records')

In [43]:
def query_skills(keyword: str = None, equipment_type: str = None, top_n: int = 10) -> str:
    """
    Search weapon skills by keyword in name or effect, or by compatible equipment type.
    Use this for detailed skill information beyond what ashesOfWar provides.
    """
    df = skills.copy()
    
    if keyword:
        mask = (
            df['name'].str.contains(keyword, case=False, na=False) |
            df['effect'].str.contains(keyword, case=False, na=False)
        )
        df = df[mask]
    
    if equipment_type:
        df = df[df['equipament'].str.contains(equipment_type, case=False, na=False)]
    
    return df[['name', 'type', 'equipament', 'FP', 'effect', 'dlc']].head(top_n).to_json(orient='records')

In [44]:
def query_shields(
    category: str = None,
    max_weight: float = 99.0,
    stat: str = None,
    min_scaling_grade: int = 0,
    top_n: int = 10
) -> str:
    """
    Search shields by category (Small Shields, Medium Shields, Greatshields),
    weight limit, or stat scaling. Use for tanky or hybrid builds.
    """
    df = shields_upgrades.copy()
    df['upgrade'] = df['upgrade'].str.replace('\xa0', ' ', regex=False).str.strip()
    
    # Get max upgrade rows same pattern as weapons
    shield_max_upgrades = MAX_UPGRADES + ['Standard +10']
    df = df[df['upgrade'].isin(shield_max_upgrades)].copy()
    
    # Parse scaling
    df['stat_scaling_parsed'] = df['stat scaling'].apply(safe_parse)
    df = pd.concat([df, df.apply(expand_scaling, axis=1)], axis=1)
    
    # Merge with shields for category and weight
    shields['requirements_parsed'] = shields['requirements'].apply(safe_parse)
    df = df.merge(
        shields[['name', 'category', 'weight']],
        left_on='shield name', right_on='name', how='left'
    )
    
    df['weight'] = pd.to_numeric(df['weight'], errors='coerce').fillna(0)
    df = df[df['weight'] <= max_weight]
    
    if category:
        df = df[df['category'].str.contains(category, case=False, na=False)]
    
    scale_col_map = {
        'Str': 'Str_scale', 'Dex': 'Dex_scale',
        'Int': 'Int_scale', 'Fai': 'Fai_scale', 'Arc': 'Arc_scale'
    }
    if stat and stat in scale_col_map:
        col = scale_col_map[stat]
        df = df[df[col] >= min_scaling_grade]
        df = df.sort_values(col, ascending=False)
    
    return df[['shield name', 'upgrade', 'category', 'weight', 
               'Str_scale', 'Dex_scale', 'Fai_scale', 'Arc_scale']].head(top_n).to_json(orient='records')

In [45]:
def query_locations(keyword: str = None, region: str = None, top_n: int = 10) -> str:
    """
    Search locations by keyword or region to find where items, weapons, or bosses
    are located. Use this to tell the user where to find recommended items.
    """
    df = locations.copy()
    
    if region:
        df = df[df['region'].str.contains(region, case=False, na=False)]
    
    if keyword:
        mask = (
            df['name'].str.contains(keyword, case=False, na=False) |
            df['items'].fillna('').str.contains(keyword, case=False, na=False) |
            df['description'].fillna('').str.contains(keyword, case=False, na=False)
        )
        df = df[mask]
    
    return df[['name', 'region', 'items', 'bosses', 'dlc']].head(top_n).to_json(orient='records')

print("5 new tools defined")

5 new tools defined


## Section 13: Updated Tool Schema and Dispatcher

Adding the five new tools to the Gemini tool schema and dispatcher so the agent
can access spirit ashes, armor, skills, shields, and item locations.

In [46]:
gemini_tools = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="query_weapons_by_stat",
            description=(
                "Search for weapons filtered by stat scaling and requirements. "
                "Use when the user specifies a primary stat (Str, Dex, Int, Fai, Arc), "
                "a weapon category, or stat caps. Returns top weapons with scaling grades and attack power."
            ),
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "stat": types.Schema(type=types.Type.STRING, description="Primary stat to sort by: 'Str', 'Dex', 'Int', 'Fai', or 'Arc'"),
                    "min_scaling_grade": types.Schema(type=types.Type.INTEGER, description="Minimum scaling grade (0=none, 1=E, 2=D, 3=C, 4=B, 5=A, 6=S)"),
                    "category": types.Schema(type=types.Type.STRING, description="Weapon category filter e.g. 'Greatsword', 'Dagger', 'Colossal'"),
                    "max_str_req": types.Schema(type=types.Type.INTEGER, description="Maximum Strength requirement"),
                    "max_dex_req": types.Schema(type=types.Type.INTEGER, description="Maximum Dexterity requirement"),
                    "max_int_req": types.Schema(type=types.Type.INTEGER, description="Maximum Intelligence requirement"),
                    "max_fai_req": types.Schema(type=types.Type.INTEGER, description="Maximum Faith requirement"),
                    "max_arc_req": types.Schema(type=types.Type.INTEGER, description="Maximum Arcane requirement"),
                    "affinity": types.Schema(type=types.Type.STRING, description="Affinity filter e.g. 'Sacred', 'Heavy', 'Blood', 'Occult'"),
                    "top_n": types.Schema(type=types.Type.INTEGER, description="Number of results to return (default 10)"),
                }
            )
        ),
        types.FunctionDeclaration(
            name="get_weapon_upgrades",
            description=(
                "Get all affinity options at max upgrade level for a specific weapon. "
                "Use this to compare how a weapon performs across different affinities. "
                "Also works for unique/somber weapons like Blasphemous Blade, Rivers of Blood, Moonveil."
            ),
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "weapon_name": types.Schema(type=types.Type.STRING, description="The name of the weapon to look up e.g. 'Greatsword', 'Rivers of Blood'"),
                },
                required=["weapon_name"]
            )
        ),
        types.FunctionDeclaration(
            name="query_talismans",
            description=(
                "Search for talismans by keyword in name, effect, or description. "
                "Use this to find talismans that complement a build e.g. 'faith', 'strength', "
                "'stamina', 'FP', 'charged attack'."
            ),
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "keyword": types.Schema(type=types.Type.STRING, description="Keyword to search in talisman name, effect, or description"),
                    "top_n": types.Schema(type=types.Type.INTEGER, description="Number of results to return (default 10)"),
                }
            )
        ),
        types.FunctionDeclaration(
            name="query_ashes_of_war",
            description=(
                "Search for Ashes of War by affinity or skill name keyword. "
                "Use this to find weapon skills that match the build's affinity or playstyle."
            ),
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "affinity": types.Schema(type=types.Type.STRING, description="Affinity type e.g. 'Sacred', 'Heavy', 'Blood', 'Occult'"),
                    "skill_keyword": types.Schema(type=types.Type.STRING, description="Keyword to search in skill name e.g. 'Storm', 'Flame', 'Gravity'"),
                    "top_n": types.Schema(type=types.Type.INTEGER, description="Number of results to return (default 10)"),
                }
            )
        ),
        types.FunctionDeclaration(
            name="query_spells",
            description=(
                "Search incantations or sorceries by stat requirements and keyword. "
                "Use this for spellcaster or hybrid builds to find usable spells."
            ),
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "spell_type": types.Schema(type=types.Type.STRING, description="'incantation' for faith/arc spells, 'sorcery' for int spells"),
                    "max_fai": types.Schema(type=types.Type.INTEGER, description="Maximum Faith requirement"),
                    "max_int": types.Schema(type=types.Type.INTEGER, description="Maximum Intelligence requirement"),
                    "max_arc": types.Schema(type=types.Type.INTEGER, description="Maximum Arcane requirement"),
                    "keyword": types.Schema(type=types.Type.STRING, description="Keyword to search in spell name or effect"),
                    "top_n": types.Schema(type=types.Type.INTEGER, description="Number of results to return (default 10)"),
                }
            )
        ),
        types.FunctionDeclaration(
            name="query_spirit_ashes",
            description=(
                "Search spirit ash summons by keyword in name or effect, with optional FP/HP cost filters. "
                "Use this to recommend summons that complement the build's playstyle — "
                "e.g. tanky summons for glass cannon builds, aggressive summons for defensive builds."
            ),
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "keyword": types.Schema(type=types.Type.STRING, description="Keyword to search in spirit ash name, effect, or description"),
                    "max_fp": types.Schema(type=types.Type.INTEGER, description="Maximum FP cost filter"),
                    "max_hp": types.Schema(type=types.Type.INTEGER, description="Maximum HP cost filter"),
                    "top_n": types.Schema(type=types.Type.INTEGER, description="Number of results to return (default 10)"),
                }
            )
        ),
        types.FunctionDeclaration(
            name="query_armor",
            description=(
                "Search armor pieces by type (helm, chest, gauntlets, leg), weight limit, "
                "or keyword in name or special effect. Use to recommend armor that complements "
                "a build — e.g. armor with special effects that boost faith, strength, or sorceries."
            ),
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "armor_type": types.Schema(type=types.Type.STRING, description="Armor slot: 'helm', 'chest', 'gauntlets', or 'leg'"),
                    "keyword": types.Schema(type=types.Type.STRING, description="Keyword in armor name, special effect, or description"),
                    "max_weight": types.Schema(type=types.Type.NUMBER, description="Maximum weight for the armor piece"),
                    "top_n": types.Schema(type=types.Type.INTEGER, description="Number of results to return (default 10)"),
                }
            )
        ),
        types.FunctionDeclaration(
            name="query_skills",
            description=(
                "Search weapon skills by keyword in name or effect description, "
                "or by compatible equipment type. Use for detailed skill info "
                "beyond what ashes of war provides — FP costs, charge attacks, compatibility."
            ),
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "keyword": types.Schema(type=types.Type.STRING, description="Keyword in skill name or effect e.g. 'storm', 'heal', 'gravity'"),
                    "equipment_type": types.Schema(type=types.Type.STRING, description="Equipment compatibility filter e.g. 'colossal', 'sword', 'polearm'"),
                    "top_n": types.Schema(type=types.Type.INTEGER, description="Number of results to return (default 10)"),
                }
            )
        ),
        types.FunctionDeclaration(
            name="query_shields",
            description=(
                "Search shields by category (Small Shields, Medium Shields, Greatshields), "
                "weight limit, or stat scaling. Use for tanky or hybrid builds that want "
                "a shield recommendation alongside their weapon."
            ),
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "category": types.Schema(type=types.Type.STRING, description="Shield category e.g. 'Small Shields', 'Medium Shields', 'Greatshields'"),
                    "max_weight": types.Schema(type=types.Type.NUMBER, description="Maximum weight for the shield"),
                    "stat": types.Schema(type=types.Type.STRING, description="Stat scaling to filter by: 'Str', 'Dex', 'Int', 'Fai', 'Arc'"),
                    "min_scaling_grade": types.Schema(type=types.Type.INTEGER, description="Minimum scaling grade (0-6)"),
                    "top_n": types.Schema(type=types.Type.INTEGER, description="Number of results to return (default 10)"),
                }
            )
        ),
        types.FunctionDeclaration(
            name="query_locations",
            description=(
                "Search locations by keyword or region to find where items, weapons, "
                "or bosses are located. Use this to tell the user where to find "
                "recommended weapons, talismans, or armor pieces."
            ),
            parameters=types.Schema(
                type=types.Type.OBJECT,
                properties={
                    "keyword": types.Schema(type=types.Type.STRING, description="Item, weapon, or boss name to search for in location data"),
                    "region": types.Schema(type=types.Type.STRING, description="Region name filter e.g. 'Limgrave', 'Mountaintops', 'Gravesite Plain'"),
                    "top_n": types.Schema(type=types.Type.INTEGER, description="Number of results to return (default 10)"),
                }
            )
        ),
    ]
)

print(f"{len(gemini_tools.function_declarations)} tools defined")

10 tools defined


In [47]:
def dispatch_tool(tool_name: str, tool_input: dict) -> str:
    dispatch_map = {
        "query_weapons_by_stat": query_weapons_by_stat,
        "get_weapon_upgrades": get_weapon_upgrades,
        "query_talismans": query_talismans,
        "query_ashes_of_war": query_ashes_of_war,
        "query_spells": query_spells,
        "query_spirit_ashes": query_spirit_ashes,
        "query_armor": query_armor,
        "query_skills": query_skills,
        "query_shields": query_shields,
        "query_locations": query_locations,
    }
    
    if tool_name in dispatch_map:
        return dispatch_map[tool_name](**tool_input)
    return json.dumps({"error": f"Unknown tool: {tool_name}"})

print("Dispatcher updated")

Dispatcher updated


In [49]:
def query_weapons_by_stat(
    stat: str = None,
    min_scaling_grade: int = 0,
    category: str = None,
    max_str_req: int = 99,
    max_dex_req: int = 99,
    max_int_req: int = 99,
    max_fai_req: int = 99,
    max_arc_req: int = 99,
    affinity: str = None,
    top_n: int = 10
) -> str:
    df = weapons_max.copy()

    # requirements_parsed and category are already in weapons_max from the rebuild merge
    def meets_reqs(req_dict):
        if not isinstance(req_dict, dict):
            return True
        return (
            req_dict.get('Str', 0) <= max_str_req and
            req_dict.get('Dex', 0) <= max_dex_req and
            req_dict.get('Int', 0) <= max_int_req and
            req_dict.get('Fai', 0) <= max_fai_req and
            req_dict.get('Arc', 0) <= max_arc_req
        )

    df = df[df['requirements_parsed'].apply(meets_reqs)]

    if affinity:
        df = df[df['upgrade'].str.startswith(affinity)]

    if category:
        df = df[df['category'].str.contains(category, case=False, na=False)]

    scale_col_map = {
        'Str': 'Str_scale', 'Dex': 'Dex_scale',
        'Int': 'Int_scale', 'Fai': 'Fai_scale', 'Arc': 'Arc_scale'
    }
    if stat and stat in scale_col_map:
        col = scale_col_map[stat]
        df = df[df[col] >= min_scaling_grade]
        df = df.sort_values(col, ascending=False)
    else:
        df = df.sort_values('phys_attack', ascending=False)

    cols = ['weapon name', 'upgrade', 'phys_attack', 'Str_scale',
            'Dex_scale', 'Int_scale', 'Fai_scale', 'Arc_scale', 'category']
    return df[cols].head(top_n).to_json(orient='records')

print("query_weapons_by_stat updated")

query_weapons_by_stat updated


Let's retry the original test now to see if there are any differences

In [50]:
result = run_build_agent(
    "I'm building a faith/strength character with 60 faith and 40 strength. "
    "I want a colossal weapon or greatsword. Recommend me a complete build "
    "including armor, a spirit ash summon, and where I can find the key items."
)


--- Turn 1 ---
Tool call: query_weapons_by_stat({'max_str_req': 40, 'max_fai_req': 60, 'stat': 'Fai', 'category': 'Greatsword'})
Result preview: [{"weapon name":"Golden Order Greatsword","upgrade":"Standard +10","phys_attack":210.0,"Str_scale":1,"Dex_scale":2,"Int_scale":0,"Fai_scale":4,"Arc_scale":0,"category":"Greatswords"},{"weapon name":"Dismounter","upgrade":"Flame +25","phys_attack":264.0,"Str_scale":1,"Dex_scale":1,"Int_scale":0,"Fai_...
Tool call: query_weapons_by_stat({'category': 'Colossal Sword', 'max_str_req': 40, 'stat': 'Fai', 'max_fai_req': 60})
Result preview: [{"weapon name":"Watchdog's Greatsword","upgrade":"Sacred +25","phys_attack":274.0,"Str_scale":3,"Dex_scale":1,"Int_scale":0,"Fai_scale":4,"Arc_scale":0,"category":"Colossal Swords"},{"weapon name":"Maliketh's Black Blade","upgrade":"Standard +10","phys_attack":311.0,"Str_scale":4,"Dex_scale":2,"Int...
Tool call: query_weapons_by_stat({'max_str_req': 40, 'max_fai_req': 60, 'stat': 'Fai', 'category': 'Colossal Wea

In [51]:
result = run_build_agent(
    "I want to do a bleed/arcane build with 60 arcane and 20 dexterity. "
    "I prefer fast weapons like daggers, curved swords, or katanas. "
    "Recommend a complete build including armor, spirit ash, and where to find key items."
)


--- Turn 1 ---
Tool call: query_weapons_by_stat({'max_dex_req': 20, 'category': 'Dagger', 'stat': 'Arc'})
Result preview: [{"weapon name":"Great Knife","upgrade":"Occult +25","phys_attack":162.0,"Str_scale":1,"Dex_scale":2,"Int_scale":0,"Fai_scale":0,"Arc_scale":5,"category":"Daggers"},{"weapon name":"Erdsteel Dagger","upgrade":"Occult +25","phys_attack":144.0,"Str_scale":1,"Dex_scale":1,"Int_scale":0,"Fai_scale":2,"Ar...
Tool call: query_weapons_by_stat({'stat': 'Arc', 'category': 'Curved Sword', 'max_dex_req': 20})
Result preview: [{"weapon name":"Scimitar","upgrade":"Occult +25","phys_attack":227.0,"Str_scale":1,"Dex_scale":2,"Int_scale":0,"Fai_scale":0,"Arc_scale":4,"category":"Curved Swords"},{"weapon name":"Falchion","upgrade":"Occult +25","phys_attack":234.0,"Str_scale":2,"Dex_scale":2,"Int_scale":0,"Fai_scale":0,"Arc_sc...
Tool call: query_weapons_by_stat({'stat': 'Arc', 'max_dex_req': 20, 'category': 'Katana'})
Result preview: [{"weapon name":"Uchigatana","upgrade":"Occult +2

Observations:
- Prioritized Occult affinity over Blood for a 60 Arcane build — that's the correct game knowledge, and it got there from the data not from memory
- Found Lord of Blood's Exultation and White Mask specifically — those are the two canonical bleed build talismans
- Recommended Mimic Tear with the correct reasoning — it copies your build so it also triggers bleed
- Pulled real location data for Lord of Blood's Exultation (Leyndell Catacombs, Esgar boss) and Mimic Tear (Nokron)
- Produced a location table at the end

## Section 14: Build Test Suite

A set of diverse test cases demonstrating the agent across different playstyles,
stat spreads, and build archetypes.

In [52]:
test_cases = [
    {
        "name": "Faith/Strength — Holy Knight",
        "prompt": (
            "I'm building a faith/strength character with 60 faith and 40 strength. "
            "I want a colossal weapon or greatsword. Recommend a complete build "
            "including armor, a spirit ash summon, and where to find key items."
        )
    },
    {
        "name": "Arcane/Dexterity — Bleed",
        "prompt": (
            "I want to do a bleed/arcane build with 60 arcane and 20 dexterity. "
            "I prefer fast weapons like daggers, curved swords, or katanas. "
            "Recommend a complete build including armor, spirit ash, and where to find key items."
        )
    },
    {
        "name": "Intelligence — Pure Sorcerer",
        "prompt": (
            "I want a pure intelligence sorcerer build with 70 intelligence and 20 mind. "
            "I want a staff and a backup melee weapon. Recommend spells, talismans, "
            "armor, a spirit ash, and where to find key items."
        )
    },
    {
        "name": "Dexterity — Samurai/Blademaster",
        "prompt": (
            "I want a pure dexterity build with 80 dexterity. "
            "I prefer katanas or curved swords, fast playstyle, no spells. "
            "Recommend a complete build with armor, spirit ash, and item locations."
        )
    }
]

print(f"{len(test_cases)} test cases defined")

4 test cases defined


In [53]:
# Run all test cases and store results
test_results = {}

for test in test_cases:
    print(f"\n{'='*60}")
    print(f"TEST: {test['name']}")
    print('='*60)
    result = run_build_agent(test['prompt'], verbose=False)
    test_results[test['name']] = result
    print(result)


TEST: Faith/Strength — Holy Knight
For a Faith/Strength build with **60 Faith** and **40 Strength**, you are perfectly positioned to use some of the most powerful weapons in the game. This stat spread favors weapons that have high base physical damage but scale aggressively with Faith for their special skills.

### 1. Primary Weapon Recommendations

Based on your stats, here are the top three choices:

*   **Blasphemous Blade (Greatsword):**
    *   **Why:** This is widely considered one of the best weapons in the game. It scales with Str (C) and Fai (B). Its skill, *Taker's Flames*, deals massive fire damage that scales purely with Faith and heals you for 10% of your Max HP + 150 on every hit.
    *   **Location:** Obtained by trading the **Remembrance of the Blasphemous** (dropped by Rykard in Mt. Gelmir) with Enia at Roundtable Hold.
*   **Maliketh's Black Blade (Colossal Sword):**
    *   **Why:** It has a B scaling in Strength and B in Faith at +10. Its skill, *Destined Death*, r

## Section 15: Project Summary

### Architecture

This project implements a **multi-tool AI agent** using the Gemini API's function
calling feature over a structured Elden Ring dataset.

The agent follows a **ReAct-style loop** (Reason → Act → Observe → Repeat):
1. The user submits a natural language build request
2. Gemini reasons about which tools to call based on the request
3. Each tool executes a pandas query over the cleaned CSV data
4. Results are fed back into the conversation
5. The agent iterates until it has enough data to produce a final recommendation

### Tools (10 total)
| Tool | Data Source | Purpose |
|---|---|---|
| `query_weapons_by_stat` | weapons_upgrades.csv | Filter weapons by scaling and requirements |
| `get_weapon_upgrades` | weapons_upgrades.csv | Compare affinities for a specific weapon |
| `query_talismans` | talismans.csv | Find talismans by keyword or effect |
| `query_ashes_of_war` | ashesOfWar.csv | Find weapon skills by affinity or name |
| `query_spells` | incantations.csv, sorceries.csv | Find spells by stat requirement |
| `query_spirit_ashes` | spiritAshes.csv | Find summons by cost or keyword |
| `query_armor` | armors.csv | Find armor by type, weight, or special effect |
| `query_skills` | skills.csv | Detailed skill info and compatibility |
| `query_shields` | shields_upgrades.csv | Find shields by category or scaling |
| `query_locations` | locations.csv | Find where items and bosses are located |

### Key Engineering Decisions
- **Somber weapons fix:** The dataset mislabeled unique weapons (Blasphemous Blade,
  Rivers of Blood, etc.) as `Standard +10` instead of `Somber +10`. Fixed by including
  `Standard +10` in the max upgrade filter alongside `Standard +25`.
- **Stringified dict parsing:** Attack power, stat scaling, and requirements were stored
  as Python dict strings. Used `ast.literal_eval` to parse them into usable structures.
- **Scaling grade mapping:** Letter grades (S/A/B/C/D/E) converted to numeric (6-1)
  to enable filtering and sorting by scaling quality.
- **On-demand querying:** The 18MB weapons_upgrades file is never loaded into the LLM
  context. The agent queries it via pandas tools, keeping token usage minimal.

### Limitations
- Somber weapons only appear at `Standard +10` in the dataset as affinities are not
  available for unique weapons since they cannot be infused
- `bosses.csv` is incomplete per the dataset author and excluded from this project
- The locations dataset indexes by area, not by item name. Some specific item lookups
  return empty and the agent falls back to general area knowledge
- The dataset was scraped from the Fextralife wiki and may contain occasional errors

### Dataset
[Ultimate Elden Ring with Shadow of the Erdtree DLC](https://www.kaggle.com/datasets/pedroaltobelli/ultimate-elden-ring-with-shadow-of-the-erdtree-dlc)
by Pedro Altobelli — CC0 Public Domain